In [ ]:
1. SELECT TOP 1 p.PlayerName,
       SUM(per.RunsScored) / COUNT(DISTINCT per.MatchID) AS BattingAverage
FROM Performance per
JOIN Players p ON per.PlayerID = p.PlayerID
GROUP BY p.PlayerName
ORDER BY BattingAverage DESC;


explaination: TOP 1 → Returns only the player with the best average.
SUM(RunsScored) → Calculates total runs scored by each player.
COUNT(DISTINCT MatchID) → Counts the number of matches played by each player.
JOIN Players ON PlayerID → Ensures player names are included in results.
GROUP BY PlayerName → Groups data per player for aggregation.
ORDER BY BattingAverage DESC → Sorts players by highest batting average.
per is alias of performance table and p is an alias for the Players table.


In [ ]:
2.  SELECT TOP 1 Winner,
       COUNT(*) AS MatchesWon,
       (COUNT(*) * 100.0) / (SELECT COUNT(*) FROM Matches) AS WinPercentage
FROM Matches
GROUP BY Winner
ORDER BY WinPercentage DESC;


Explanation: COUNT(*) → Counts total matches won by each team.
Subquery (SELECT COUNT(*) FROM Matches) → Finds total matches played.
 (COUNT(*) * 100.0) / TotalMatches → Calculates win percentage.
 ORDER BY WinPercentage DESC → Sorts teams by highest win percentage.
 TOP 1 → Returns only the team with the best win percentage.

In [ ]:
3. SELECT TOP 1 p.PlayerName, per.MatchID, per.RunsScored,
       (per.RunsScored * 100.0) / SUM(per.RunsScored) OVER (PARTITION BY per.MatchID) AS ContributionPercentage
FROM Performance per
JOIN Players p ON per.PlayerID = p.PlayerID
ORDER BY ContributionPercentage DESC;

Explanation: - SELECT TOP 1
- Retrieves only the first row after sorting.
- p.PlayerName, per.MatchID, per.RunsScored
- Selects relevant data: the player’s name, the match they played in, and the number of runs they scored.
- (per.RunsScored * 100.0) / SUM(per.RunsScored) OVER (PARTITION BY per.MatchID) AS ContributionPercentage
- Calculates what percentage of total runs in a match was scored by this player.
- Uses a window function to sum RunsScored for each match (PARTITION BY per.MatchID).
- Multiplies by 100.0 to get a percentage and ensure floating-point division.
- FROM Performance per JOIN Players p ON per.PlayerID = p.PlayerID
- Joins Performance and Players tables to link each performance to a player name.
- ORDER BY ContributionPercentage DESC
- Sorts the results by ContributionPercentage in descending order—putting the highest contributor at the top.



In [ ]:
4. with playerstats as (
    select playerid, stdev(runsScored) as runs_std_dev
    from performance
    group by playerid
)
SELECT p.PlayerName, ps.runs_std_dev
FROM PlayerStats ps
JOIN Players p ON ps.PlayerID = p.PlayerID
ORDER BY ps.runs_std_dev ASC;

Explaination: - WITH PlayerStats AS
- This is a Common Table Expression (CTE) that creates a temporary result set named PlayerStats to use in the main query.
- SELECT PlayerID, STDEV(RunsScored) AS runs_std_dev
- Calculates the standard deviation of RunsScored for each player.
- STDEV() is a statistical function that measures how much a player's scores vary from their average—lower values mean more consistent performance.
- FROM Performance GROUP BY PlayerID
- Groups the data by player so that the standard deviation is calculated per individual.
- Main Query: SELECT p.PlayerName, ps.runs_std_dev
- Retrieves each player’s name along with their run standard deviation by joining the PlayerStats CTE with the Players table.
- JOIN Players p ON ps.PlayerID = p.PlayerID
- Matches player stats to their names using a foreign key.
- ORDER BY ps.runs_std_dev ASC
- Sorts the results so that the most consistent players (lowest variation in runs) appear first.



In [ ]:
5.  SELECT MatchID,
       SUM(RunsScored) AS TotalRuns,
       SUM(WicketsTaken) AS TotalWickets,
       SUM(Catches) AS TotalCatches,
       (SUM(RunsScored) + SUM(WicketsTaken) + SUM(Catches)) AS CombinedTotal
FROM Performance
GROUP BY MatchID
HAVING (SUM(RunsScored) + SUM(WicketsTaken) + SUM(Catches)) > 500;

Explaination: - SELECT MatchID
- Identifies each unique match from the Performance table.
- SUM(RunsScored) AS TotalRuns
- Calculates the total number of runs scored by all players in each match.
- SUM(WicketsTaken) AS TotalWickets
- Totals up the wickets taken by all players in each match.
- SUM(Catches) AS TotalCatches
- Adds up all catches taken in a match.
- (SUM(RunsScored) + SUM(WicketsTaken) + SUM(Catches)) AS CombinedTotal
- Combines all three metrics to calculate a single overall performance metric for the match.
- FROM Performance GROUP BY MatchID
- Aggregates data on a per-match basis—each row represents one match.
- HAVING CombinedTotal > 500
- Filters the results to include only matches where the sum of runs, wickets, and catches exceeds 500.
- Uses HAVING because it filters aggregated results (unlike WHERE, which filters before aggregation).


In [ ]:
6. WITH RankedPlayers AS (
    SELECT PlayerID, MatchID, RunsScored, WicketsTaken,
           RANK() OVER (PARTITION BY MatchID ORDER BY RunsScored DESC, WicketsTaken DESC) AS Rank
    FROM Performance
)
SELECT TOP 1 p.PlayerName, COUNT(*) AS PlayerOfMatchAwards
FROM RankedPlayers rp
JOIN Players p ON rp.PlayerID = p.PlayerID
WHERE rp.Rank = 1
GROUP BY p.PlayerName
ORDER BY COUNT(*) DESC;

Explaination:  CTE: WITH RankedPlayers AS (...)
- Ranks players within each match (PARTITION BY MatchID)
- Uses RANK() function to assign rank based on:
- RunsScored (descending)
- WicketsTaken (descending) as a tiebreaker
- So, the player with the most runs, and in case of a tie, the most wickets, is given Rank 1.
 Main Query
- SELECT TOP 1 p.PlayerName, COUNT(*) AS PlayerOfMatchAwards
- Finds the player who achieved Rank 1 in the most number of matches
- Effectively counts how many times a player was the top performer in a match
- JOIN Players p ON rp.PlayerID = p.PlayerID
- Links player stats to their names
- WHERE rp.Rank = 1
- Filters to only include top performers per match (like awarding “Player of the Match”)
- GROUP BY p.PlayerName
- Groups by player name to count their total number of top performances
- ORDER BY COUNT(*) DESC
- Sorts in descending order to get the player with the most awards at the top



In [ ]:
7. WITH PlayerRoleCounts AS (
    SELECT PlayerID, PlayerName, COUNT(DISTINCT Role) AS UniqueRoles
    FROM Players
    GROUP BY PlayerID, PlayerName
)
SELECT PlayerID, PlayerName, UniqueRoles,
       NTILE(4) OVER (ORDER BY UniqueRoles DESC) AS DiversityRank
FROM PlayerRoleCounts
ORDER BY DiversityRank;

Explaination: CTE: PlayerRoleCounts
- SELECT PlayerID, PlayerName, COUNT(DISTINCT Role) AS UniqueRoles
- Counts the number of distinct roles each player has performed.
- Helps measure role diversity (e.g., player A has played as batsman, bowler, and fielder → count = 3).
- GROUP BY PlayerID, PlayerName
- Ensures aggregation is done per individual player.
 Main Query
- NTILE(4) OVER (ORDER BY UniqueRoles DESC) AS DiversityRank
- Divides all players into 4 equal groups (quartiles) based on UniqueRoles, from most diverse (1) to least (4).
- Players with similar role counts end up in the same diversity tier.
- ORDER BY DiversityRank
- Final result is sorted by diversity tier, showing most versatile players first.



In [ ]:
8. WITH TeamRuns AS (
    SELECT m.MatchID, m.Team1, m.Team2,
           SUM(CASE WHEN pl.TeamName = m.Team1 THEN p.RunsScored ELSE 0 END) AS Team1Runs,
           SUM(CASE WHEN pl.TeamName = m.Team2 THEN p.RunsScored ELSE 0 END) AS Team2Runs
    FROM Matches m
    JOIN Performance p ON m.MatchID = p.MatchID
    JOIN Players pl ON p.PlayerID = pl.PlayerID  -- Linking Players table to get TeamName
    GROUP BY m.MatchID, m.Team1, m.Team2
)
SELECT MatchID, Team1, Team2, ABS(Team1Runs - Team2Runs) AS RunDiff
FROM TeamRuns
WHERE Team1Runs <> Team2Runs
ORDER BY RunDiff ASC;

Explaination: CTE: TeamRuns
- SELECT m.MatchID, m.Team1, m.Team2
- Retrieves each match and its two competing teams.
- SUM(CASE WHEN pl.TeamName = m.Team1 THEN p.RunsScored ELSE 0 END) AS Team1Runs
- Sums the runs scored by players belonging to Team1.
- SUM(CASE WHEN pl.TeamName = m.Team2 THEN p.RunsScored ELSE 0 END) AS Team2Runs
- Sums the runs scored by players belonging to Team2.
- FROM Matches m JOIN Performance p ON m.MatchID = p.MatchID & JOIN Players pl ON p.PlayerID = pl.PlayerID
- Connects the Matches, Performance, and Players tables to map scores to teams.
- GROUP BY m.MatchID, m.Team1, m.Team2
- Groups by match to calculate totals per team per match.
 Main Query
- SELECT MatchID, Team1, Team2, ABS(Team1Runs - Team2Runs) AS RunDiff
- Selects match info and calculates the absolute difference in runs between the two teams.
- WHERE Team1Runs <> Team2Runs
- Filters out tied matches (where both teams scored the same).
- ORDER BY RunDiff ASC
- Sorts by smallest run difference first, so the closest contests appear at the top.


In [ ]:
9. SELECT p.PlayerID
FROM Performance p
JOIN Matches m ON p.MatchID = m.MatchID
GROUP BY p.PlayerID
HAVING COUNT(DISTINCT p.MatchID) = (SELECT COUNT(DISTINCT MatchID) FROM Matches);

Explaination: - SELECT p.PlayerID
- The query aims to return PlayerIDs of those players who meet the criteria.
- FROM Performance p JOIN Matches m ON p.MatchID = m.MatchID
- Joins the Performance table with the Matches table so that every player's match participation is lined up with existing matches.
- GROUP BY p.PlayerID
- Aggregates performance data per player to count how many matches they’ve participated in.
- HAVING COUNT(DISTINCT p.MatchID)
- Counts how many distinct matches each player has appeared in.
- DISTINCT ensures a player isn’t counted multiple times for repeated entries in the same match.
- = (SELECT COUNT(DISTINCT MatchID) FROM Matches)
- Compares that count to the total number of distinct matches in the tournament or season.
- Only selects players whose number of matches exactly equals the total number of matches.


o/p is blank, coz there is no matching playerid from performance's matchid and matches' matchid.

In [ ]:
10. SELECT TOP 1
    m.MatchID,
    m.Team1,
    m.Team2,
    ABS(
        SUM(CASE WHEN pl.TeamName = m.Team1 THEN per.RunsScored ELSE 0 END) -
        SUM(CASE WHEN pl.TeamName = m.Team2 THEN per.RunsScored ELSE 0 END)
    ) AS RunMargin
FROM Matches m
JOIN Performance per ON m.MatchID = per.MatchID
JOIN Players pl ON per.PlayerID = pl.PlayerID
GROUP BY m.MatchID, m.Team1, m.Team2
ORDER BY RunMargin ASC;

Explaination: - SELECT TOP 1 m.MatchID, m.Team1, m.Team2
- Retrieves details about a match: its ID and the two competing teams.
- TOP 1 ensures only the closest match is selected.
- SUM(CASE WHEN pl.TeamName = m.Team1 THEN per.RunsScored ELSE 0 END)
- Sums all runs scored by players from Team1 in the match.
- SUM(CASE WHEN pl.TeamName = m.Team2 THEN per.RunsScored ELSE 0 END)
- Sums all runs scored by players from Team2 in the match.
- ABS(...) AS RunMargin
- Calculates the absolute difference between the two teams’ run totals.
- This avoids negative values and simply reflects how close the match was.
- FROM Matches m JOIN Performance per ON m.MatchID = per.MatchID JOIN Players pl ON per.PlayerID = pl.PlayerID
- Joins:
- Matches to Performance: links each match to performances.
- Performance to Players: gets each player’s team for the match.
- GROUP BY m.MatchID, m.Team1, m.Team2
- Groups records by match so total team scores can be calculated per match.
- ORDER BY RunMargin ASC
- Sorts matches from smallest to largest run margin.


In [ ]:
11. SELECT pl.TeamName, SUM(per.RunsScored) AS TotalRuns
FROM Performance per
JOIN Players pl ON per.PlayerID = pl.PlayerID
GROUP BY pl.TeamName
ORDER BY TotalRuns DESC;

Explaination: - SELECT pl.TeamName
- Retrieves the name of each team by accessing the TeamName field from the Players table.
- SUM(per.RunsScored) AS TotalRuns
- Calculates the total runs scored by all players belonging to a team.
- Uses SUM() to add up every player’s RunsScored across all matches.
- FROM Performance per JOIN Players pl ON per.PlayerID = pl.PlayerID
- Joins the Performance and Players tables to:
- Match each performance record to the correct player
- Access the player’s team information
- GROUP BY pl.TeamName
- Groups the aggregated run totals by team—so each row in the output represents one team.
- ORDER BY TotalRuns DESC
- Sorts the results by TotalRuns in descending order—so the highest-scoring team appears first.



In [ ]:
12. SELECT m.MatchID, m.Winner, SUM(per.WicketsTaken) AS TotalWickets
FROM Matches m
JOIN Performance per ON m.MatchID = per.MatchID
GROUP BY m.MatchID, m.Winner
HAVING SUM(per.WicketsTaken) > 2
ORDER BY TotalWickets DESC;

Explaination: - SELECT m.MatchID, m.Winner, SUM(per.WicketsTaken) AS TotalWickets
- Retrieves:
- MatchID: the unique identifier for the match
- Winner: the team that won the match
- TotalWickets: total wickets taken in that match (by summing performances)
- FROM Matches m JOIN Performance per ON m.MatchID = per.MatchID
- Joins the Matches and Performance tables so each match’s statistics can be calculated using player performance data.
- GROUP BY m.MatchID, m.Winner
- Groups data by match and winner to allow aggregation (i.e., summing wickets) per match.
- HAVING SUM(per.WicketsTaken) > 2
- Filters to include only matches where the total number of wickets taken exceeds 2.
- HAVING is used here because it filters results after aggregation (as opposed to WHERE).
- ORDER BY TotalWickets DESC
- Sorts the output so that matches with the most total wickets come first.


In [ ]:
13. SELECT TOP 5
    per.MatchID,
    pl.PlayerName,
    per.RunsScored
FROM Performance per
JOIN Players pl ON per.PlayerID = pl.PlayerID
ORDER BY per.RunsScored DESC;

Explaination: - SELECT TOP 5
- Limits the result to only the first 5 rows after sorting.
- per.MatchID
- Displays the match in which the performance occurred.
- pl.PlayerName
- Retrieves the name of the player who scored the runs.
- per.RunsScored
- Shows the number of runs scored by the player in that match.
- FROM Performance per JOIN Players pl ON per.PlayerID = pl.PlayerID
- Joins the Performance table (which contains the actual match stats) with the Players table to get player names linked to their stats.
- ORDER BY per.RunsScored DESC
- Sorts all individual performances in descending order of runs scored, so the top-scoring innings appear first.



In [ ]:
14. SELECT pl.Role, pl.PlayerName, SUM(per.WicketsTaken) AS TotalWickets
FROM Players pl
JOIN Performance per ON pl.PlayerID = per.PlayerID
WHERE pl.Role = 'Bowler'
GROUP BY pl.Role, pl.PlayerName
HAVING SUM(per.WicketsTaken) >= 5
ORDER BY TotalWickets DESC;

Explaination: - SELECT pl.Role, pl.PlayerName, SUM(per.WicketsTaken) AS TotalWickets
- Retrieves:
- The player's role (which will be 'Bowler' due to the filter)
- The player's name
- The total number of wickets they’ve taken by summing up WicketsTaken across all their performances
- FROM Players pl JOIN Performance per ON pl.PlayerID = per.PlayerID
- Joins the Players table with the Performance table to:
- Match player details with their match statistics
- WHERE pl.Role = 'Bowler'
- Filters the dataset to include only players whose role is 'Bowler'
- GROUP BY pl.Role, pl.PlayerName
- Aggregates data by player name and role so the SUM function can calculate per player
- HAVING SUM(per.WicketsTaken) >= 5
- Filters the grouped results to show only those bowlers who have taken 5 or more wickets total
- ORDER BY TotalWickets DESC
- Sorts the final list so the top-performing bowlers (most wickets) are shown first


In [ ]:
15. SELECT
    m.MatchID,
    m.Winner AS WinningTeam,
    SUM(per.Catches) AS TotalCatches
FROM Matches m
JOIN Performance per ON m.MatchID = per.MatchID
GROUP BY m.MatchID, m.Winner
ORDER BY m.MatchID;

Explaination: - SELECT m.MatchID
- Retrieves the unique identifier for each match.
- m.Winner AS WinningTeam
- Fetches the winning team’s name and labels the column as WinningTeam.
- SUM(per.Catches) AS TotalCatches
- Totals up the number of catches taken by all players in that match using the Performance table.
- FROM Matches m JOIN Performance per ON m.MatchID = per.MatchID
- Joins the Matches and Performance tables:
- Matches holds info like match IDs and results.
- Performance contains individual player stats including Catches.
- GROUP BY m.MatchID, m.Winner
- Groups the records by both match and winner, so the aggregation (SUM) is computed for each distinct match-winner combination.
- ORDER BY m.MatchID
- Sorts the results in ascending order of match ID for organized presentation.


In [ ]:
16. SELECT TOP 1
    pl.PlayerName,
    COUNT(DISTINCT per.MatchID) AS MatchesPlayed,
    SUM(per.RunsScored * 1.5 + per.WicketsTaken * 25 +
        per.Catches * 10 + per.Stumpings * 15 + per.RunOuts * 10) AS TotalImpactScore
FROM Players pl
JOIN Performance per ON pl.PlayerID = per.PlayerID
GROUP BY pl.PlayerName
HAVING COUNT(DISTINCT per.MatchID) >= 3
ORDER BY TotalImpactScore DESC;

explaination: - SELECT TOP 1
- Limits the result to just one player—the one with the highest impact score.
- pl.PlayerName
- Retrieves the name of the player.
- COUNT(DISTINCT per.MatchID) AS MatchesPlayed
- Counts how many unique matches the player has played in.
- DISTINCT ensures repeated entries from the same match don't inflate the count.
- SUM(...) AS TotalImpactScore
- Calculates a custom impact score based on individual performance:
- This weighted sum rewards multi-dimensional contributions across batting, bowling, and fielding.
- FROM Players pl JOIN Performance per ON pl.PlayerID = per.PlayerID
- Joins the Players and Performance tables to bring together player info and match stats.
- GROUP BY pl.PlayerName
- Aggregates stats per player so SUM and COUNT calculations are done for each individual.
- HAVING COUNT(DISTINCT per.MatchID) >= 3
- Filters out players who participated in fewer than 3 matches.
- Ensures that the impact score reflects sustained contributions, not one-off performances.
- ORDER BY TotalImpactScore DESC
- Ranks players by their overall impact, putting the highest scorer at the top.



In [ ]:
17. SELECT top 1
    m.MatchID,
    m.Team1,
    m.Team2, m.winner,
    ABS(
        SUM(CASE WHEN pl.TeamName = m.Team1 THEN per.RunsScored ELSE 0 END) -
        SUM(CASE WHEN pl.TeamName = m.Team2 THEN per.RunsScored ELSE 0 END)
    ) AS RunMargin
FROM Matches m
JOIN Performance per ON m.MatchID = per.MatchID
JOIN Players pl ON per.PlayerID = pl.PlayerID
where m.winner is not null
GROUP BY m.MatchID, m.Team1, m.Team2, m.winner
ORDER BY RunMargin ASC;

Explaination: - Joins the Matches, Performance, and Players tables to connect player scores to teams.
- For each match:
- Calculates total runs scored by each team using conditional SUM(CASE ...).
- Finds the absolute difference in scores between the two teams (RunMargin).
- Filters only the matches where a winner is declared (m.winner IS NOT NULL).
- Orders matches by RunMargin in ascending order.
- Returns only the top result, i.e. the match with the smallest winning margin.


In [ ]:
18. SELECT p.PlayerName
FROM Players p
JOIN Performance perf ON p.PlayerID = perf.PlayerID
JOIN Matches m ON perf.MatchID = m.MatchID
WHERE perf.RunsScored = (
    SELECT MAX(p2.RunsScored)
    FROM Performance p2
    JOIN Players p3 ON p2.PlayerID = p3.PlayerID
    WHERE p2.MatchID = perf.MatchID AND p3.TeamName = p.TeamName
)
GROUP BY p.PlayerID, p.PlayerName
HAVING COUNT(*) > (
    SELECT COUNT(*)/2.0
    FROM Performance pf
    JOIN Players pl ON pf.PlayerID = pl.PlayerID
    WHERE pf.PlayerID = p.PlayerID
);

explaination: This joins Players, Performance, and Matches tables so we know:
- Who played,
- In which match,
- For what team (via Players.TeamName),
- And how many runs they scored (perf.RunsScored).
- In each match (perf.MatchID),
- For the player's team (p.TeamName),
- Whether the player (p.PlayerID) had the maximum runs scored among teammates.
If that's true, the player was the top scorer on their team in that match.

Now for each player:
- COUNT(*) = number of matches where they were the top scorer.
- We check if this count is greater than half the total matches they played.
So only players who led their team in scoring in more than 50% of their games are selected.

Final Result
The output will list players like:
- Virat Kohli if he was the top run-scorer for India in more than half of his matches.
- Same logic for other players.


In [ ]:
19. WITH PlayerImpact AS (
    SELECT
        perf.PlayerID,
        pl.PlayerName,
        COUNT(*) AS MatchesPlayed,
        SUM(perf.RunsScored * 1.5 + perf.WicketsTaken * 25 +
            perf.Catches * 10 + perf.Stumpings * 15 + perf.RunOuts * 10) AS TotalImpact
    FROM Performance perf
    JOIN Players pl ON perf.PlayerID = pl.PlayerID
    GROUP BY perf.PlayerID, pl.PlayerName
),
FilteredPlayers AS (
    SELECT *,
           TotalImpact * 1.0 / MatchesPlayed AS AvgImpact
    FROM PlayerImpact
    WHERE MatchesPlayed >= 3
)
SELECT
    PlayerName,
    MatchesPlayed,
    ROUND(AvgImpact, 2) AS AvgImpact,
    DENSE_RANK() OVER (ORDER BY AvgImpact DESC) AS ImpactRank
FROM FilteredPlayers
ORDER BY ImpactRank;

explaination :- Joins the Players and Performance tables.
- Calculates MatchesPlayed (via COUNT(*)).
- Computes TotalImpact using your custom formula(Impact = Runs × 1.5 + Wickets × 25 + Catches × 10 + Stumpings × 15 + RunOuts × 10).
- Filters players to include only those with 3+ matches.
- Calculates AvgImpact per match (TotalImpact / MatchesPlayed).
The * 1.0 ensures it's a floating-point division, not integer.
- Lists each qualified player’s name, match count, and average impact (rounded to 2 decimal places).
- Uses DENSE_RANK() to assign ranks. Ties are grouped together (e.g., two players with the same score will share Rank 1, and the next gets Rank 2).




In [ ]:
20. WITH MatchTotals AS (
    SELECT
        m.MatchID,
        m.Team1,
        m.Team2,
        SUM(per.RunsScored) AS TotalRuns
    FROM Performance per
    JOIN Matches m ON per.MatchID = m.MatchID
    GROUP BY m.MatchID, m.Team1, m.Team2
),
RankedMatches AS (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY TotalRuns DESC) AS MatchRank
    FROM MatchTotals
)
SELECT MatchID, Team1, Team2, TotalRuns
FROM RankedMatches
WHERE MatchRank <= 3
ORDER BY MatchRank;

Explaination: This query identifies the top 3 matches with the highest total runs scored by both teams combined.
It first calculates each match’s total runs by summing all individual player scores from the Performance table, joined with match details from the Matches table.
 Then, it uses the DENSE_RANK() window function to rank matches by TotalRuns in descending order, ensuring that ties receive the same rank.
 Finally, it filters the result to include only matches ranked in the top 3 and presents each match with its participating teams and total runs.
 This approach is both accurate and rank-aware, ideal for distinguishing truly high-scoring games even when run totals are tied.



In [ ]:
21. SELECT
    per.PlayerID,
    m.MatchDate,
    per.MatchID,
    (per.RunsScored * 1.5 + per.WicketsTaken * 25 +
     per.Catches * 10 + per.Stumpings * 15 + per.RunOuts * 10) AS Impact,
    SUM(
        per.RunsScored * 1.5 + per.WicketsTaken * 25 +
        per.Catches * 10 + per.Stumpings * 15 + per.RunOuts * 10
    ) OVER (
        PARTITION BY per.PlayerID
        ORDER BY m.MatchDate, per.MatchID
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS CumulativeImpact
FROM Performance per
JOIN Matches m ON per.MatchID = m.MatchID
WHERE per.PlayerID IN (
    SELECT PlayerID
    FROM Performance
    GROUP BY PlayerID
    HAVING COUNT(*) >= 3
)
ORDER BY per.PlayerID, m.MatchDate, per.MatchID;


Explaination: Calculate Impact per Match
(per.RunsScored * 1.5 + per.WicketsTaken * 25 +
 per.Catches * 10 + per.Stumpings * 15 + per.RunOuts * 10) AS Impact

Window function for cumulative impact:
- It partitions by player, so the calculation resets for each player.
- Within each player’s data, it orders matches chronologically (by date and ID).
- Then, it adds up each match’s impact progressively, giving a running total.
Filter Players With 3+ Matches

Final Output and Order
The result shows:
- Player ID
- Match date and ID
- Impact for that match
- Cumulative impact up to that match
All sorted by player and match date, which gives you a smooth timeline of each player's performance growth.





